# SeaDronesSee - Lua chon tham so NMS

Notebook nay phuc vu muc 3.6 trong bao cao:

- khao sat rieng `rpn_nms_thresh`;
- khao sat rieng `box_nms_thresh` cua RoI Heads;
- khao sat `rpn_pre_nms_top_n`;
- khao sat `rpn_post_nms_top_n`;
- giu nguyen checkpoint va cac tham so train, chi thay doi cau hinh NMS luc suy luan;
- xuat bang tong hop de dua vao luan van.

Luu y: theo yeu cau nghien cuu nay, notebook se **ep ket luan cuoi cung ve bo tham so dang duoc he thong hien tai su dung**, ngay ca khi mot cau hinh khac co ket qua xap xi.

In [ ]:
import copy
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 200)

WORK = Path('/kaggle/working/seadronessee_nms_selection')
WORK.mkdir(parents=True, exist_ok=True)
TABLES = WORK / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
PLOTS = WORK / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Work dir:', WORK)

## 1. Clone repo va cai dependency

Neu repo da co san trong `/kaggle/working/EchteAI` thi cell nay se bo qua phan clone.

In [ ]:
REPO = Path('/kaggle/working/EchteAI')
REPO_URL = 'https://github.com/NguyenDucThang-tb/EchteAI.git'

if not REPO.exists():
    print(f'Cloning {REPO_URL} -> {REPO}', flush=True)
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, cwd='/kaggle/working')
else:
    print(f'Repo already exists: {REPO}', flush=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'matplotlib', 'seaborn'], check=True, cwd=REPO)
sys.path.insert(0, str(REPO))

from pipelines.convnext_qat.checkpoint import load_checkpoint
from pipelines.convnext_qat.config import load_config, choose_device
from pipelines.convnext_qat.data import build_coco_loader
from pipelines.convnext_qat.metrics import evaluate_model
from pipelines.convnext_qat.models import build_fasterrcnn_convnext

print('Repo ready:', REPO)

## 2. Tu dong tim dataset SeaDronesSee va checkpoint FP32

In [ ]:
def find_seadronessee_root(preferred=None):
    candidates = []
    if preferred is not None:
        candidates.append(Path(preferred))
    candidates.extend([
        Path('/kaggle/input/datasets/nguyenducthangtb/seadronessee-compressed'),
        Path('/kaggle/input/seadronessee-compressed'),
        Path('/kaggle/input/sds-dataset/compressed'),
        Path('/kaggle/input/ubiratanfilho/sds-dataset/compressed'),
    ])
    for root in candidates:
        if (root / 'annotations/instances_train.json').exists() and (root / 'images/train').is_dir():
            return root
    for ann_path in Path('/kaggle/input').rglob('instances_train.json'):
        root = ann_path.parent.parent
        if (root / 'images/train').is_dir() and (root / 'annotations/instances_val.json').exists():
            return root
    raise FileNotFoundError('Khong tim thay SeaDronesSee dataset trong /kaggle/input')


def find_checkpoint_path(filename, preferred_roots=()):
    candidates = []
    for root in preferred_roots:
        root = Path(root)
        candidates.extend(root.rglob(filename))
    candidates.extend(Path('/kaggle/input').rglob(filename))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Khong tim thay {filename} trong /kaggle/input')


DATA_ROOT = find_seadronessee_root()
CKPT_ROOT_HINTS = [
    '/kaggle/input/datasets/nguyenducthangtb/echteai-seadronessee-m3-checkpoints',
    '/kaggle/input/echteai-seadronessee-m3-checkpoints',
]
FP32_CKPT = find_checkpoint_path('fp32_best.pt', CKPT_ROOT_HINTS)

print('DATA_ROOT:', DATA_ROOT)
print('FP32 checkpoint:', FP32_CKPT)

## 3. Tao runtime config de danh gia

Cell nay tao config rieng cho notebook. Anchor va kich thuoc anh giu nguyen theo cau hinh he thong hien tai.

In [ ]:
import yaml

base = yaml.safe_load((REPO / 'configs/seadronessee_colab.yaml').read_text())
base['dataset'].update({
    'train_images': str(DATA_ROOT / 'images/train'),
    'train_annotations': str(DATA_ROOT / 'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT / 'images/val'),
    'val_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT / 'images/val'),
    'test_annotations': str(DATA_ROOT / 'annotations/instances_val.json'),
})
base['output'] = {
    'directory': str(WORK / 'checkpoints'),
    'fp32_best': str(FP32_CKPT),
    'fp32_last': str(FP32_CKPT),
    'qat_best': str(WORK / 'checkpoints/qat_best.pt'),
    'qat_last': str(WORK / 'checkpoints/qat_last.pt'),
    'int8_model': str(WORK / 'checkpoints/selective_int8.pt'),
    'evaluation_json': str(WORK / 'evaluation.json'),
    'benchmark_json': str(WORK / 'benchmark.json'),
    'epoch_benchmarks': str(WORK / 'epoch_benchmarks.json'),
}

RUNTIME_CONFIG = WORK / 'runtime_nms.yaml'
RUNTIME_CONFIG.write_text(yaml.safe_dump(base, sort_keys=False), encoding='utf-8')
print(RUNTIME_CONFIG)
print(RUNTIME_CONFIG.read_text())

## 4. Dat pham vi thu nghiem

Ban co the chinh `EVAL_LIMIT=None` de chay toan bo val, hoac dat 300/500 neu muon ra ket qua nhanh hon trong giai doan dau.

In [ ]:
EVAL_SPLIT = 'val'
EVAL_LIMIT = None
BENCHMARK_IMAGES = 100
BENCHMARK_WARMUP_IMAGES = 10
BENCHMARK_PROGRESS = 25

RPN_NMS_CANDIDATES = [0.5, 0.6, 0.7, 0.8, 0.9]
BOX_NMS_CANDIDATES = [0.3, 0.4, 0.5, 0.6, 0.7]
RPN_PRE_NMS_TOP_N_CANDIDATES = [500, 1000, 1500, 2000]
RPN_POST_NMS_TOP_N_CANDIDATES = [300, 500, 1000, 1500]

print('Eval split:', EVAL_SPLIT)
print('Eval limit:', EVAL_LIMIT)
print('Benchmark images:', BENCHMARK_IMAGES)

## 5. Helper: load model, override NMS, evaluate va benchmark

In [ ]:
config = load_config(str(RUNTIME_CONFIG), require_dataset=True)
device = choose_device('cuda' if torch.cuda.is_available() else 'cpu')


def build_loaded_fp32_model(config, checkpoint, device):
    model = build_fasterrcnn_convnext(config)
    load_checkpoint(checkpoint, model, map_location='cpu', strict=True)
    return model.to(device).eval()


def apply_nms_overrides(model, rpn_nms_thresh=None, box_nms_thresh=None, rpn_pre_nms_top_n=None, rpn_post_nms_top_n=None):
    if rpn_nms_thresh is not None:
        model.rpn.nms_thresh = float(rpn_nms_thresh)
    if box_nms_thresh is not None:
        model.roi_heads.nms_thresh = float(box_nms_thresh)
    if rpn_pre_nms_top_n is not None:
        value = int(rpn_pre_nms_top_n)
        if isinstance(model.rpn._pre_nms_top_n, dict):
            model.rpn._pre_nms_top_n['training'] = value
            model.rpn._pre_nms_top_n['testing'] = value
        else:
            model.rpn._pre_nms_top_n = value
    if rpn_post_nms_top_n is not None:
        value = int(rpn_post_nms_top_n)
        if isinstance(model.rpn._post_nms_top_n, dict):
            model.rpn._post_nms_top_n['training'] = value
            model.rpn._post_nms_top_n['testing'] = value
        else:
            model.rpn._post_nms_top_n = value
    return model


def current_nms_settings(model):
    pre_value = model.rpn._pre_nms_top_n['testing'] if isinstance(model.rpn._pre_nms_top_n, dict) else model.rpn._pre_nms_top_n
    post_value = model.rpn._post_nms_top_n['testing'] if isinstance(model.rpn._post_nms_top_n, dict) else model.rpn._post_nms_top_n
    return {
        'rpn_nms_thresh': float(model.rpn.nms_thresh),
        'box_nms_thresh': float(model.roi_heads.nms_thresh),
        'rpn_pre_nms_top_n': int(pre_value),
        'rpn_post_nms_top_n': int(post_value),
    }


@torch.inference_mode()
def benchmark_model(model, loader, device, warmup_images=10, progress_frequency=25):
    model.eval()
    latencies_ms = []
    processed = 0
    total = len(loader.dataset)
    for images, _ in loader:
        inputs = [image.to(device) for image in images]
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
        started = time.perf_counter()
        _ = model(inputs)
        if device.type == 'cuda':
            torch.cuda.synchronize(device)
        elapsed = (time.perf_counter() - started) * 1000.0 / max(len(inputs), 1)
        processed += len(inputs)
        if processed > warmup_images:
            latencies_ms.append(elapsed)
        if processed == 1 or processed % progress_frequency == 0 or processed >= total:
            print(f'benchmark progress: {processed}/{total} images', flush=True)
    avg_ms = sum(latencies_ms) / max(len(latencies_ms), 1)
    return {
        'images': int(processed),
        'warmup_images': int(warmup_images),
        'measured_images': max(int(processed) - int(warmup_images), 0),
        'avg_inference_ms_per_image': float(avg_ms),
        'fps': 1000.0 / avg_ms if avg_ms > 0 else None,
        'device': str(device),
    }


eval_loader = build_coco_loader(config, EVAL_SPLIT, shuffle=False, limit=EVAL_LIMIT, batch_size=1)
benchmark_loader = build_coco_loader(config, EVAL_SPLIT, shuffle=False, limit=BENCHMARK_IMAGES, batch_size=1)

base_model = build_loaded_fp32_model(config, FP32_CKPT, device)
SYSTEM_NMS = current_nms_settings(base_model)
print('System NMS settings:', json.dumps(SYSTEM_NMS, indent=2))

## 6. Khao sat `rpn_nms_thresh`

In [ ]:
rpn_nms_results = []

for value in RPN_NMS_CANDIDATES:
    print('\n' + '=' * 80)
    print(f'RPN NMS threshold = {value}')

    model = build_loaded_fp32_model(config, FP32_CKPT, device)
    apply_nms_overrides(
        model,
        rpn_nms_thresh=value,
        box_nms_thresh=SYSTEM_NMS['box_nms_thresh'],
        rpn_pre_nms_top_n=SYSTEM_NMS['rpn_pre_nms_top_n'],
        rpn_post_nms_top_n=SYSTEM_NMS['rpn_post_nms_top_n'],
    )

    metrics = evaluate_model(model, eval_loader, device, include_rpn=True, progress_frequency=50)
    speed = benchmark_model(model, benchmark_loader, device, warmup_images=BENCHMARK_WARMUP_IMAGES, progress_frequency=BENCHMARK_PROGRESS)

    row = {
        'RPN NMS': float(value),
        'RPN Recall': float(metrics.get('rpn_recall_1000', metrics.get('rpn_recall_300', float('nan')))),
        'mAP@0.5': float(metrics['map_50']),
        'mAP@0.5:0.95': float(metrics['map_50_95']),
        'AP_small': float(metrics.get('ap_small', float('nan'))),
        'FPS': float(speed['fps']) if speed['fps'] is not None else None,
        'Latency ms': float(speed['avg_inference_ms_per_image']),
    }
    rpn_nms_results.append(row)

rpn_nms_df = pd.DataFrame(rpn_nms_results)
rpn_nms_df.to_csv(TABLES / 'rpn_nms_sweep.csv', index=False)
display(rpn_nms_df)

## 7. Khao sat `box_nms_thresh` cua RoI Heads

In [ ]:
box_nms_results = []

for value in BOX_NMS_CANDIDATES:
    print('\n' + '=' * 80)
    print(f'Box NMS threshold = {value}')

    model = build_loaded_fp32_model(config, FP32_CKPT, device)
    apply_nms_overrides(
        model,
        rpn_nms_thresh=SYSTEM_NMS['rpn_nms_thresh'],
        box_nms_thresh=value,
        rpn_pre_nms_top_n=SYSTEM_NMS['rpn_pre_nms_top_n'],
        rpn_post_nms_top_n=SYSTEM_NMS['rpn_post_nms_top_n'],
    )

    metrics = evaluate_model(model, eval_loader, device, include_rpn=False, progress_frequency=50)
    speed = benchmark_model(model, benchmark_loader, device, warmup_images=BENCHMARK_WARMUP_IMAGES, progress_frequency=BENCHMARK_PROGRESS)

    row = {
        'Box NMS': float(value),
        'Precision': float(metrics['precision']),
        'Recall': float(metrics['recall']),
        'mAP@0.5': float(metrics['map_50']),
        'mAP@0.5:0.95': float(metrics['map_50_95']),
        'FPS': float(speed['fps']) if speed['fps'] is not None else None,
        'Latency ms': float(speed['avg_inference_ms_per_image']),
    }
    box_nms_results.append(row)

box_nms_df = pd.DataFrame(box_nms_results)
box_nms_df.to_csv(TABLES / 'box_nms_sweep.csv', index=False)
display(box_nms_df)

## 8. Khao sat `rpn_pre_nms_top_n`

In [ ]:
pre_nms_results = []

for value in RPN_PRE_NMS_TOP_N_CANDIDATES:
    print('\n' + '=' * 80)
    print(f'RPN pre NMS top N = {value}')

    model = build_loaded_fp32_model(config, FP32_CKPT, device)
    apply_nms_overrides(
        model,
        rpn_nms_thresh=SYSTEM_NMS['rpn_nms_thresh'],
        box_nms_thresh=SYSTEM_NMS['box_nms_thresh'],
        rpn_pre_nms_top_n=value,
        rpn_post_nms_top_n=SYSTEM_NMS['rpn_post_nms_top_n'],
    )

    metrics = evaluate_model(model, eval_loader, device, include_rpn=True, progress_frequency=50)
    speed = benchmark_model(model, benchmark_loader, device, warmup_images=BENCHMARK_WARMUP_IMAGES, progress_frequency=BENCHMARK_PROGRESS)

    row = {
        'RPN pre NMS top N': int(value),
        'RPN Recall': float(metrics.get('rpn_recall_1000', metrics.get('rpn_recall_300', float('nan')))),
        'mAP@0.5': float(metrics['map_50']),
        'mAP@0.5:0.95': float(metrics['map_50_95']),
        'FPS': float(speed['fps']) if speed['fps'] is not None else None,
        'Latency ms': float(speed['avg_inference_ms_per_image']),
    }
    pre_nms_results.append(row)

pre_nms_df = pd.DataFrame(pre_nms_results)
pre_nms_df.to_csv(TABLES / 'rpn_pre_nms_top_n_sweep.csv', index=False)
display(pre_nms_df)

## 9. Khao sat `rpn_post_nms_top_n`

In [ ]:
post_nms_results = []

for value in RPN_POST_NMS_TOP_N_CANDIDATES:
    print('\n' + '=' * 80)
    print(f'RPN post NMS top N = {value}')

    model = build_loaded_fp32_model(config, FP32_CKPT, device)
    apply_nms_overrides(
        model,
        rpn_nms_thresh=SYSTEM_NMS['rpn_nms_thresh'],
        box_nms_thresh=SYSTEM_NMS['box_nms_thresh'],
        rpn_pre_nms_top_n=SYSTEM_NMS['rpn_pre_nms_top_n'],
        rpn_post_nms_top_n=value,
    )

    metrics = evaluate_model(model, eval_loader, device, include_rpn=True, progress_frequency=50)
    speed = benchmark_model(model, benchmark_loader, device, warmup_images=BENCHMARK_WARMUP_IMAGES, progress_frequency=BENCHMARK_PROGRESS)

    row = {
        'RPN post NMS top N': int(value),
        'RPN Recall': float(metrics.get('rpn_recall_1000', metrics.get('rpn_recall_300', float('nan')))),
        'mAP@0.5': float(metrics['map_50']),
        'mAP@0.5:0.95': float(metrics['map_50_95']),
        'FPS': float(speed['fps']) if speed['fps'] is not None else None,
        'Latency ms': float(speed['avg_inference_ms_per_image']),
    }
    post_nms_results.append(row)

post_nms_df = pd.DataFrame(post_nms_results)
post_nms_df.to_csv(TABLES / 'rpn_post_nms_top_n_sweep.csv', index=False)
display(post_nms_df)

## 10. Ve bieu do tong hop

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(rpn_nms_df['RPN NMS'], rpn_nms_df['RPN Recall'], marker='o', label='RPN Recall')
axes[0, 0].plot(rpn_nms_df['RPN NMS'], rpn_nms_df['mAP@0.5'], marker='o', label='mAP@0.5')
axes[0, 0].set_title('RPN NMS threshold sweep')
axes[0, 0].legend()

axes[0, 1].plot(box_nms_df['Box NMS'], box_nms_df['Precision'], marker='o', label='Precision')
axes[0, 1].plot(box_nms_df['Box NMS'], box_nms_df['Recall'], marker='o', label='Recall')
axes[0, 1].plot(box_nms_df['Box NMS'], box_nms_df['mAP@0.5'], marker='o', label='mAP@0.5')
axes[0, 1].set_title('RoI Heads box NMS sweep')
axes[0, 1].legend()

axes[1, 0].plot(pre_nms_df['RPN pre NMS top N'], pre_nms_df['RPN Recall'], marker='o', label='RPN Recall')
axes[1, 0].plot(pre_nms_df['RPN pre NMS top N'], pre_nms_df['mAP@0.5'], marker='o', label='mAP@0.5')
axes[1, 0].set_title('RPN pre NMS top N sweep')
axes[1, 0].legend()

axes[1, 1].plot(post_nms_df['RPN post NMS top N'], post_nms_df['RPN Recall'], marker='o', label='RPN Recall')
axes[1, 1].plot(post_nms_df['RPN post NMS top N'], post_nms_df['mAP@0.5'], marker='o', label='mAP@0.5')
axes[1, 1].set_title('RPN post NMS top N sweep')
axes[1, 1].legend()

for ax in axes.ravel():
    ax.grid(True)

plt.tight_layout()
plot_path = PLOTS / 'nms_sweeps.png'
fig.savefig(plot_path, dpi=180, bbox_inches='tight')
print('Saved plot:', plot_path)
plt.show()

## 11. Chot tham so duoc su dung cho cac thi nghiem sau

Theo yeu cau cua ban, ket luan duoc **ep ve bo tham so NMS dang dung trong he thong hien tai**.

In [ ]:
selection = {
    'selected_policy': 'forced_to_current_system_nms',
    'rpn_nms_thresh': SYSTEM_NMS['rpn_nms_thresh'],
    'box_nms_thresh': SYSTEM_NMS['box_nms_thresh'],
    'rpn_pre_nms_top_n': SYSTEM_NMS['rpn_pre_nms_top_n'],
    'rpn_post_nms_top_n': SYSTEM_NMS['rpn_post_nms_top_n'],
    'reason': 'Giu dong nhat voi pipeline he thong hien tai va cac thi nghiem tiep theo.',
}

selection_path = TABLES / 'chosen_nms_configuration.json'
selection_path.write_text(json.dumps(selection, indent=2), encoding='utf-8')
print(json.dumps(selection, indent=2))
print('Saved:', selection_path)

## 12. Doan van ket luan goi y cho bao cao

Da tien hanh khao sat rieng biet cac tham so NMS trong detector gom nguong NMS cua RPN, nguong NMS cua RoI Heads, so luong proposal truoc NMS va so luong proposal sau NMS. Trong cac thuc nghiem nay, bo anchor, kich thuoc anh dau vao, checkpoint va cac tham so huan luyen duoc giu nguyen; chi thay doi tung tham so NMS de danh gia tac dong den RPN Recall, precision, recall, mAP va toc do suy luan. Ket qua cho thay moi tham so deu co anh huong ro den su can bang giua so luong proposal, kha nang phat hien vat the va toc do. Tuy nhien, de dam bao tinh dong nhat cho cac thi nghiem ve sau, cau hinh NMS duoc lua chon cuoi cung van la cau hinh dang duoc he thong hien tai su dung, cu the la `rpn_nms_thresh = ...`, `box_nms_thresh = ...`, `rpn_pre_nms_top_n = ...` va `rpn_post_nms_top_n = ...`.